# 04 Data Preprocessing & SMOTE Balancing

## Objective
Prepare the dataset for model training:
- Standardize images to 224x224 RGB format
- Create stratified Train (80%) / Val (10%) / Test (10%) splits
- Stream preprocessed images as compressed NPZ files (<100 MB RAM usage)
- Generate SMOTE balanced dataset for multi-class balancing across all models
- Export label encoding, split indices, and class weights

**Prerequisites:** `01_data_check.ipynb`, `02_dataset_visualization.ipynb`, `03_targeted_augmentation.ipynb`
**Next Step:** `05_cnn_baseline.ipynb`

### Import Libraries

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import sys
import os

sys.path.insert(0, r"D:\cnn_model\utils")  # Adjust the path as needed')

from data_utils import (
    verify_dataset_structure,
    get_image_files,
    load_image,
    create_image_label_mapping,
    denoise_image,
    enhance_image_clahe,
    preprocess_image_complete,
    check_image_quality
)
from model_utils import (
    create_data_augmentation,
    create_validation_generator,
    compute_class_weights_from_labels
)
from visualization_utils import (
    plot_sample_images,
    save_plot
)

import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import pickle
import json
import random
from tqdm import tqdm

# Set random seeds
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

print("✓ Libraries imported successfully!")


✓ Libraries imported successfully!


### Configuration

In [8]:
# Configuration
DATASET_PATH = r"D:\cnn_model\Augmented_images"  # Adjust the path as needed
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 16
TRAIN_SPLIT = 0.8  # Increased from 0.7 to 0.8
VAL_SPLIT = 0.10    # Reduced to 10%
TEST_SPLIT = 0.10   # Reduced to 10%

print("Updated Configuration:")
print(f"  Image Size: {IMAGE_SIZE}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Train/Val/Test Split: {TRAIN_SPLIT}/{VAL_SPLIT}/{TEST_SPLIT}")

# Load dataset info
stats = verify_dataset_structure(DATASET_PATH)
print(f"\n✓ Dataset loaded: {stats['total_images']} images")

Updated Configuration:
  Image Size: (224, 224)
  Batch Size: 16
  Train/Val/Test Split: 0.8/0.1/0.1

✓ Dataset loaded: 8600 images


### 1. Create Image-Label Mapping

In [9]:
print("Creating image-label mapping...\n")

image_labels, label_encoding = create_image_label_mapping(DATASET_PATH)

print(f"Total images mapped: {len(image_labels)}")
print(f"Total classes: {len(label_encoding)}")
print(f"\nClass Encoding:")
for label, idx in sorted(label_encoding.items(), key=lambda x: x[1]):
    count = sum(1 for v in image_labels.values() if v == label)
    print(f"  {idx:2d} -> {label:40s} | {count:4d} images")

Creating image-label mapping...

Total images mapped: 8600
Total classes: 40

Class Encoding:
   0 -> Brinjal_Healthy                          |  290 images
   1 -> Brinjal_Early_Leaf_spot                  |  315 images
   2 -> Brinjal_Eggplant Wilt Disease            |  295 images
   3 -> Brinjal_Leaf_curl                        |  200 images
   4 -> Brinjal_Little_leaf                      |  200 images
   5 -> Brinjal_Magnessium                       |  200 images
   6 -> Brinjal_Mosaic                           |  200 images
   7 -> Brinjal_Nitrogen                         |  200 images
   8 -> Brinjal_pest_damage                      |  265 images
   9 -> Brinjal_Pottasium                        |  200 images
  10 -> Castor_Healthy                           |  200 images
  11 -> Castor_Alternia Leaf Blight              |  205 images
  12 -> Castor_Bacterial Leaf Blight             |  200 images
  13 -> Castor_Cercospera Leaf Spot              |  200 images
  14 -> Castor_Peast_dis

### 2. PREPROCESS IMAGES & CREATE TRAIN/VAL/TEST SPLIT

**What happens:**
- Creates 80/10/10 stratified train/val/test split (happens ONCE here)
- Loads each image, preprocesses (denoise + enhance + sharpen), saves as NPZ
- Saves split metadata to JSON files

**Memory:** Only 1 image in RAM at a time ✓

In [10]:
print("=" * 70)
print("MEMORY-EFFICIENT PREPROCESSING")
print("=" * 70)
print("\n✓ Strategy: Load → Preprocess → Save (one at a time)")
print("✓ Avoids loading entire dataset into RAM")
print("✓ Datasets will be created from saved files\n")

# Create output directories
output_base = r"D:\cnn_model\results\preprocessed_data_efficient"
os.makedirs(f"{output_base}/train_images", exist_ok=True)
os.makedirs(f"{output_base}/val_images", exist_ok=True)
os.makedirs(f"{output_base}/test_images", exist_ok=True)

print(f"Output directory: {output_base}\n")

# First pass: create metadata and indices WITHOUT loading all images
image_paths = list(image_labels.keys())
image_y = []

print("Step 1: Creating dataset split and saving metadata...\n")

# Create train/val/test split BEFORE loading images
from sklearn.model_selection import train_test_split

# Prepare indices for stratified split
indices = np.arange(len(image_paths))
paths_array = np.array(image_paths)
labels_array = np.array([label_encoding[image_labels[p]] for p in image_paths])

# First split: train (80%) + temp (20%)
train_idx, temp_idx, train_labels, temp_labels = train_test_split(
    indices, labels_array,
    train_size=TRAIN_SPLIT,
    test_size=(VAL_SPLIT + TEST_SPLIT),
    random_state=42,
    stratify=labels_array
)

# Second split: val and test
val_relative_size = VAL_SPLIT / (VAL_SPLIT + TEST_SPLIT)
val_idx, test_idx = train_test_split(
    temp_idx,
    train_size=val_relative_size,
    random_state=42
)

split_info = {
    'train_indices': train_idx.tolist(),
    'val_indices': val_idx.tolist(),
    'test_indices': test_idx.tolist(),
    'train_count': len(train_idx),
    'val_count': len(val_idx),
    'test_count': len(test_idx)
}

with open(f'{output_base}/split_indices.json', 'w') as f:
    json.dump(split_info, f, indent=4)

print(f"✓ Train split: {len(train_idx)} images")
print(f"✓ Val split: {len(val_idx)} images")
print(f"✓ Test split: {len(test_idx)} images")
print(f"✓ Saved split indices\n")

# Save image paths and labels
paths_labels = {
    'image_paths': image_paths,
    'labels': labels_array.tolist()
}
with open(f'{output_base}/image_metadata.json', 'w') as f:
    json.dump(paths_labels, f, indent=4)

print(f"✓ Saved image metadata\n")

# Step 2: Process images one-by-one and save
print("Step 2: Loading and preprocessing images (saving individually)...\n")
print("This processes images in batches and saves to disk\n")

failed_images = []

def process_and_save_image(img_path, output_path, img_idx, total):
    """Load, preprocess, and save a single image."""
    try:
        # Load
        img = load_image(img_path, target_size=IMAGE_SIZE)
        if img is None:
            return False
        
        # Normalize
        img = img.astype('float32') / 255.0
        
        # Preprocess (advanced pipeline)
        processed_img, _ = preprocess_image_complete(
            img,
            denoise=True,
            enhance=True,
            sharpen=True,
            check_quality=True  # Skip quality check for speed
        )
        
        # Save as compressed NPZ
        np.savez_compressed(output_path, image=processed_img)
        return True
    except Exception as e:
        print(f"  ✗ Error: {img_path}")
        return False

# Process and save each image
print("Processing TRAINING images...")
for i, idx in enumerate(tqdm(train_idx, desc="Train")):
    img_path = image_paths[idx]
    output_file = f"{output_base}/train_images/train_{i:05d}.npz"
    success = process_and_save_image(img_path, output_file, i, len(train_idx))
    if not success:
        failed_images.append(img_path)

print(f"\nProcessing VALIDATION images...")
for i, idx in enumerate(tqdm(val_idx, desc="Val")):
    img_path = image_paths[idx]
    output_file = f"{output_base}/val_images/val_{i:05d}.npz"
    success = process_and_save_image(img_path, output_file, i, len(val_idx))
    if not success:
        failed_images.append(img_path)

print(f"\nProcessing TEST images...")
for i, idx in enumerate(tqdm(test_idx, desc="Test")):
    img_path = image_paths[idx]
    output_file = f"{output_base}/test_images/test_{i:05d}.npz"
    success = process_and_save_image(img_path, output_file, i, len(test_idx))
    if not success:
        failed_images.append(img_path)

print(f"\n✓ Preprocessing complete!")
print(f"✓ Successfully saved: {len(train_idx) + len(val_idx) + len(test_idx) - len(failed_images)} images")
print(f"⚠️  Failed: {len(failed_images)} images")

if failed_images:
    print("\nFailed images:")
    for img in failed_images[:5]:
        print(f"  - {img}")

print(f"\n✓ All preprocessed images saved to: {output_base}")

MEMORY-EFFICIENT PREPROCESSING

✓ Strategy: Load → Preprocess → Save (one at a time)
✓ Avoids loading entire dataset into RAM
✓ Datasets will be created from saved files

Output directory: D:\cnn_model\results\preprocessed_data_efficient

Step 1: Creating dataset split and saving metadata...

✓ Train split: 6880 images
✓ Val split: 860 images
✓ Test split: 860 images
✓ Saved split indices

✓ Saved image metadata

Step 2: Loading and preprocessing images (saving individually)...

This processes images in batches and saves to disk

Processing TRAINING images...


MEMORY-EFFICIENT PREPROCESSING

✓ Strategy: Load → Preprocess → Save (one at a time)
✓ Avoids loading entire dataset into RAM
✓ Datasets will be created from saved files

Output directory: D:\cnn_model\results\preprocessed_data_efficient

Step 1: Creating dataset split and saving metadata...

✓ Train split: 6880 images
✓ Val split: 860 images
✓ Test split: 860 images
✓ Saved split indices

✓ Saved image metadata

Step 2: Loading and preprocessing images (saving individually)...

This processes images in batches and saves to disk

Processing TRAINING images...


Train: 100%|██████████| 6880/6880 [14:50<00:00,  7.72it/s]


MEMORY-EFFICIENT PREPROCESSING

✓ Strategy: Load → Preprocess → Save (one at a time)
✓ Avoids loading entire dataset into RAM
✓ Datasets will be created from saved files

Output directory: D:\cnn_model\results\preprocessed_data_efficient

Step 1: Creating dataset split and saving metadata...

✓ Train split: 6880 images
✓ Val split: 860 images
✓ Test split: 860 images
✓ Saved split indices

✓ Saved image metadata

Step 2: Loading and preprocessing images (saving individually)...

This processes images in batches and saves to disk

Processing TRAINING images...


Train: 100%|██████████| 6880/6880 [14:50<00:00,  7.72it/s]


...


Processing VALIDATION images...


Val: 100%|██████████| 860/860 [01:56<00:00,  7.36it/s]



Processing TEST images...


Test: 100%|██████████| 860/860 [02:03<00:00,  6.96it/s]


✓ Preprocessing complete!
✓ Successfully saved: 8600 images
⚠️  Failed: 0 images

✓ All preprocessed images saved to: D:\cnn_model\results\preprocessed_data_efficient


### 3. LOAD METADATA & CREATE ONE-HOT LABELS

**What happens:**
- Loads split indices from JSON (saved in step 2)
- Loads image metadata from JSON (saved in step 2)
- Creates one-hot encoded labels for training

**Key Point:** NO NEW SPLIT - uses what was already saved in step 2 ✓

In [11]:
print("Loading metadata and creating one-hot encoded labels...\n")

Loading metadata and creating one-hot encoded labels...

✓ Loaded split indices:
  - Train: 6880 images
  - Val: 860 images
  - Test: 860 images

✓ Loaded image metadata: 8600 images

✓ Number of classes: 40

✓ Created one-hot encoded labels:
  - y_train shape: (6880, 40)
  - y_val shape: (860, 40)
  - y_test shape: (860, 40)

✓ Step 3 complete!


### 4. DATA AUGMENTATION CONFIGURATION

**What happens:**
- Defines augmentation parameters for training
- Augmentation is applied DURING TRAINING (not here)
- Validation/Test: NO augmentation

In [12]:
print("\nData Augmentation Configuration (Applied During Training)\n")
print("=" * 70)
print("✓ On-the-fly augmentation will be applied during training")
print("✓ No augmentation for validation/test sets")
print("\nAugmentation Parameters (for training):")
print(f"  - Rotation: ±20°")
print(f"  - Width shift: ±20%")
print(f"  - Height shift: ±20%")
print(f"  - Shear: ±0.2")
print(f"  - Zoom: ±20%")
print(f"  - Horizontal flip: Yes")
print("\n✓ Augmentation is applied dynamically during each epoch")
print("✓ No physical augmented copies are stored (saves disk space)")
print("=" * 70)


Data Augmentation Configuration (Applied During Training)

✓ On-the-fly augmentation will be applied during training
✓ No augmentation for validation/test sets

Augmentation Parameters (for training):
  - Rotation: ±20°
  - Width shift: ±20%
  - Height shift: ±20%
  - Shear: ±0.2
  - Zoom: ±20%
  - Horizontal flip: Yes

✓ Augmentation is applied dynamically during each epoch
✓ No physical augmented copies are stored (saves disk space)


### 5. SAVE METADATA & CONFIGURATION

**What gets saved:**
- label_encoding.json (class → index mapping)
- class_weights.json (balanced weights)
- preprocessing_info.json (configuration)

In [13]:
import os
print("Saving preprocessing configuration and metadata...\n")

output_base = r"D:\cnn_model\results\preprocessed_data_efficient"
os.makedirs(output_base, exist_ok=True)

# Save label encoding
with open(f'{output_base}/label_encoding.json', 'w') as f:
    json.dump(label_encoding, f, indent=4)
print("✓ Saved label_encoding.json")

# Save class weights
class_names = [label for label, _ in sorted(label_encoding.items(), key=lambda item: item[1])]
class_weights = compute_class_weights_from_labels(y_train, class_names=class_names, verbose=False)
with open(f'{output_base}/class_weights.json', 'w') as f:
    json.dump({str(class_index): float(weight) for class_index, weight in class_weights.items()}, f, indent=4)
print("✓ Saved class_weights.json")

# Save preprocessing info
preprocess_info = {
    'image_size': IMAGE_SIZE,
    'num_classes': num_classes,
    'train_samples': split_info['train_count'],
    'val_samples': split_info['val_count'],
    'test_samples': split_info['test_count'],
    'normalization': 'Divided by 255',
    'preprocessing_pipeline': ['Bilateral Denoising', 'CLAHE Enhancement', 'Unsharp Sharpening'],
    'batch_size': BATCH_SIZE,
    'file_format': 'NPZ (compressed)',
    'storage_strategy': 'Per-image files on disk',
    'memory_efficient': True
}

with open(f'{output_base}/preprocessing_info.json', 'w') as f:
    json.dump(preprocess_info, f, indent=4)
print("✓ Saved preprocessing_info.json")

print(f"\n✓ All metadata files saved to: {output_base}")

Saving preprocessing configuration and metadata...

✓ Saved label_encoding.json
✓ Saved class_weights.json
✓ Saved preprocessing_info.json

✓ All metadata files saved to: D:\cnn_model\results\preprocessed_data_efficient


### 6. CREATE TENSORFLOW DATASETS FROM SAVED FILES

**What happens:**
- Creates generator functions to load images on-demand
- Creates train_dataset, val_dataset, test_dataset
- Memory: Only 1 batch (~100 MB) in RAM at a time ✓

In [14]:
print("Creating TensorFlow Datasets from saved files...\n")

output_base = r"D:\cnn_model\results\preprocessed_data_efficient"

# Define generator function to load images from disk
def load_image_from_file(img_idx, split_type):
    """Load preprocessed image from .npz file."""
    file_path = f"{output_base}/{split_type}_images/{split_type}_{img_idx:05d}.npz"
    try:
        data = np.load(file_path)
        return data['image'].astype('float32')
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        # Return blank image on error
        return np.zeros((224, 224, 3), dtype='float32')

# Create generators for each split
def train_generator(batch_size=BATCH_SIZE):
    """Generator for training data."""
    n_samples = split_info['train_count']
    indices = np.arange(n_samples)
    np.random.shuffle(indices)
    
    for i in range(0, n_samples, batch_size):
        batch_indices = indices[i:min(i+batch_size, n_samples)]
        batch_images = []
        batch_labels = []
        
        for idx in batch_indices:
            img = load_image_from_file(idx, 'train')
            batch_images.append(img)
            batch_labels.append(y_train[idx])
        
        yield np.array(batch_images), np.array(batch_labels)

def val_generator(batch_size=BATCH_SIZE):
    """Generator for validation data."""
    n_samples = split_info['val_count']
    
    for i in range(0, n_samples, batch_size):
        batch_indices = np.arange(i, min(i+batch_size, n_samples))
        batch_images = []
        batch_labels = []
        
        for idx in batch_indices:
            img = load_image_from_file(idx, 'val')
            batch_images.append(img)
            batch_labels.append(y_val[idx])
        
        yield np.array(batch_images), np.array(batch_labels)

def test_generator(batch_size=BATCH_SIZE):
    """Generator for test data."""
    n_samples = split_info['test_count']
    
    for i in range(0, n_samples, batch_size):
        batch_indices = np.arange(i, min(i+batch_size, n_samples))
        batch_images = []
        batch_labels = []
        
        for idx in batch_indices:
            img = load_image_from_file(idx, 'test')
            batch_images.append(img)
            batch_labels.append(y_test[idx])
        
        yield np.array(batch_images), np.array(batch_labels)

# Create tf.data Datasets from generators
train_dataset = tf.data.Dataset.from_generator(
    lambda: train_generator(BATCH_SIZE),
    output_signature=(
        tf.TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32),
        tf.TensorSpec(shape=(None, num_classes), dtype=tf.float32)
    )
)
train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_generator(
    lambda: val_generator(BATCH_SIZE),
    output_signature=(
        tf.TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32),
        tf.TensorSpec(shape=(None, num_classes), dtype=tf.float32)
    )
)
val_dataset = val_dataset.prefetch(tf.data.AUTOTUNE)

test_dataset = tf.data.Dataset.from_generator(
    lambda: test_generator(BATCH_SIZE),
    output_signature=(
        tf.TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32),
        tf.TensorSpec(shape=(None, num_classes), dtype=tf.float32)
    )
)
test_dataset = test_dataset.prefetch(tf.data.AUTOTUNE)

print(f"✓ Train dataset: ~{split_info['train_count']//BATCH_SIZE} batches")
print(f"✓ Val dataset: ~{split_info['val_count']//BATCH_SIZE} batches")
print(f"✓ Test dataset: ~{split_info['test_count']//BATCH_SIZE} batches")

# Verify a batch
print(f"\nVerifying data format...")
for batch_images, batch_labels in train_dataset.take(1):
    print(f"✓ Batch shape: {batch_images.shape}")
    print(f"✓ Labels shape: {batch_labels.shape}")
    print(f"✓ Image dtype: {batch_images.dtype}")
    print(f"✓ Image value range: [{batch_images.numpy().min():.4f}, {batch_images.numpy().max():.4f}]")

Creating TensorFlow Datasets from saved files...

✓ Train dataset: ~430 batches
✓ Val dataset: ~53 batches
✓ Test dataset: ~53 batches

Verifying data format...
✓ Batch shape: (16, 224, 224, 3)
✓ Labels shape: (16, 40)
✓ Image dtype: <dtype: 'float32'>
✓ Image value range: [0.0000, 1.0000]


### 7. FINAL SUMMARY

**Workflow completed:**
- ✓ Preprocessed 8600+ images
- ✓ Created train/val/test split (80/10/10)
- ✓ Saved NPZ files to disk
- ✓ Created one-hot encoded labels
- ✓ Created TensorFlow Datasets
- ✓ Memory-efficient: Full dataset NOT in RAM

In [15]:
print("\n" + "=" * 70)
print("✓ MEMORY-EFFICIENT DATA PREPROCESSING COMPLETE")
print("=" * 70)

output_base = r"D:\cnn_model\results\preprocessed_data_efficient"

# Summary
print(f"\n📊 Summary:")
print(f" Image size: {IMAGE_SIZE}")
print(f" Number of classes: {num_classes}")
print(f" Batch size: {BATCH_SIZE}")

# Data split
print(f"\n📈 Data Split:")
print(f" Training: {split_info['train_count']} images")
print(f" Validation: {split_info['val_count']} images")
print(f" Testing: {split_info['test_count']} images")

# Storage
print(f"\n💾 Storage Strategy:")
print(" Format: Compressed NPZ files (1 per image)")
print(f" Location: {output_base}")
print(" Subdirs: train_images/, val_images/, test_images/")

# Memory optimization
batch_memory = (
    BATCH_SIZE * IMAGE_SIZE[0] * IMAGE_SIZE[1] * 3 * 4
) / (1024 ** 2)

print(f"\n⚡ Memory Optimization:")
print(f" ✓ Only current batch in RAM (~{batch_memory:.1f} MB/batch)")
print(" ✓ Full dataset NOT loaded into RAM")
print(" ✓ Load-on-demand from disk")
print(" ✓ Prevents MemoryError at high RAM usage")

# Files created
print(f"\n📁 Files Created:")
print(f" - train_images/ ({split_info['train_count']} .npz files)")
print(f" - val_images/ ({split_info['val_count']} .npz files)")
print(f" - test_images/ ({split_info['test_count']} .npz files)")
print(" - split_indices.json")
print(" - image_metadata.json")
print(" - label_encoding.json")
print(" - class_weights.json")
print(" - preprocessing_info.json")

# Dataset status
print(f"\n✓ TensorFlow Datasets created and ready:")
print(" - train_dataset (loads from train_images/)")
print(" - val_dataset (loads from val_images/)")
print(" - test_dataset (loads from test_images/)")

# Labels
print(f"\n✓ One-hot encoded labels ready:")
print(f" - y_train shape: {y_train.shape}")
print(f" - y_val shape: {y_val.shape}")
print(f" - y_test shape: {y_test.shape}")

# Next step
print(f"\n→ Next Step: Run 05_cnn_baseline.ipynb")

print("\n" + "=" * 70)


✓ MEMORY-EFFICIENT DATA PREPROCESSING COMPLETE

📊 Summary:
 Image size: (224, 224)
 Number of classes: 40
 Batch size: 16

📈 Data Split:
 Training: 6880 images
 Validation: 860 images
 Testing: 860 images

💾 Storage Strategy:
 Format: Compressed NPZ files (1 per image)
 Location: D:\cnn_model\results\preprocessed_data_efficient
 Subdirs: train_images/, val_images/, test_images/

⚡ Memory Optimization:
 ✓ Only current batch in RAM (~9.2 MB/batch)
 ✓ Full dataset NOT loaded into RAM
 ✓ Load-on-demand from disk
 ✓ Prevents MemoryError at high RAM usage

📁 Files Created:
 - train_images/ (6880 .npz files)
 - val_images/ (860 .npz files)
 - test_images/ (860 .npz files)
 - split_indices.json
 - image_metadata.json
 - label_encoding.json
 - class_weights.json
 - preprocessing_info.json

✓ TensorFlow Datasets created and ready:
 - train_dataset (loads from train_images/)
 - val_dataset (loads from val_images/)
 - test_dataset (loads from test_images/)

✓ One-hot encoded labels ready:
 - y_tr

In [ ]:
print("\n" + "="*70)
print("✓ MEMORY-EFFICIENT DATA PREPROCESSING COMPLETE")
print("="*70)

output_base = r"D:\cnn_model\results\preprocessed_data_efficient"

print(f"\n📊 Summary:")
print(f"  Image size: {IMAGE_SIZE}")
print(f"  Number of classes: {num_classes}")
print(f"  Batch size: {BATCH_SIZE}")

print(f"\n📈 Data Split:")
print(f"  Training: {split_info['train_count']} images")
print(f"  Validation: {split_info['val_count']} images")
print(f"  Testing: {split_info['test_count']} images")
print(f"  Total: {sum(split_info.values())-1} images")  # -1 to exclude duplicate train/val/test counts

print(f"\n💾 Storage Strategy:")
print(f"  Format: Compressed NPZ files (1 per image)")
print(f"  Location: {output_base}")
print(f"  Subdirs: train_images/, val_images/, test_images/")
print(f"  Per-image: ~100-200 KB")

print(f"\n⚡ Memory Optimization:")
print(f"  ✓ Only current batch in RAM (~{BATCH_SIZE * 224 * 224 * 3 * 4 / (1024**2):.1f} MB/batch)")
print(f"  ✓ Full dataset NOT in RAM")
print(f"  ✓ Load-on-demand from disk")
print(f"  ✓ Avoids MemoryError at 98% RAM")

print(f"\n📁 Files Created:")
print(f"  - test_images/ (~{split_info['test_count']} .npz files)")
print(f"  - split_indices.json")
print(f"  - image_metadata.json")
print(f"  - label_encoding.json")
print(f"  - class_weights.json")
print(f"  - preprocessing_info.json")

print(f"\n✓ TensorFlow Datasets ready for training:")
print(f"  - train_dataset (with batching)")
print(f"  - val_dataset (with batching)")
print(f"  - test_dataset (with batching)")

print(f"\n→ Next Step: 05_cnn_baseline.ipynb")
print("="*70)